# Read in Balanced Pathway projection sectoral emissions.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr

Carbon Budgets are five-year carbon emission limits set by the UK. The Balanced Pathway is a scenario that outlines how the UK can meet its carbon budgets while achieving net-zero emissions by 2050. The Balanced Pathway projection includes sectoral and sub-sectoral emissions data.

Data can be obtained [from the CB7 website](https://www.theccc.org.uk/publication/the-seventh-carbon-budget/).

This notebook reads the Balanced Pathway sectoral and sub-sectoral emission projections data and generates xarray datasets which can be imported directly using the agrifoodpy-data package. The dataset is structured with dimensions for time and sector, and a data variable for each of the variables reported in the orginal dataset.


In [2]:
datafile = "../../data/impact/The-Seventh-Carbon-Budget-full-dataset.xlsx"

## Sector data

In [3]:
sector_level_data = pd.read_excel(datafile, sheet_name="Sector-level data")
sector_level_data.head()

,scenario,country,sector,variable,variable_unit,year,value
0,Balanced Pathway,United Kingdom,Agriculture,Emissions: direct emissions total,MtCO2e,2025,47.044319
1,Balanced Pathway,United Kingdom,Agriculture,Emissions: direct emissions total,MtCO2e,2026,46.559718
2,Balanced Pathway,United Kingdom,Agriculture,Emissions: direct emissions total,MtCO2e,2027,44.919614
3,Balanced Pathway,United Kingdom,Agriculture,Emissions: direct emissions total,MtCO2e,2028,43.081191
4,Balanced Pathway,United Kingdom,Agriculture,Emissions: direct emissions total,MtCO2e,2029,41.070621


In [4]:
sector_dataset = xr.Dataset()

for variable in sector_level_data["variable"].unique():
    variable_data = sector_level_data[sector_level_data["variable"] == variable]
    variable_data = variable_data[variable_data["scenario"] == "Balanced Pathway"]
    variable_data = variable_data.drop(columns=["variable", "scenario", "country", "variable_unit"])
    variable_ds = xr.Dataset.from_dataframe(variable_data.set_index(["year", "sector"]))
    sector_dataset[variable] = variable_ds.to_array().squeeze()

sector_dataset = sector_dataset.drop("variable")
sector_dataset = sector_dataset.rename({"year": "Year", "sector": "Sector"})

In [5]:
sector_dataset

<xarray.Dataset>
Dimensions:                                                     (Year: 26,
                                                                 Sector: 13)
Coordinates:
  * Year                                                        (Year) int64 ...
  * Sector                                                      (Sector) object ...
Data variables: (12/51)
    Emissions: direct emissions total                           (Year, Sector) float64 ...
    Emissions: direct emissions CO2                             (Year, Sector) float64 ...
    Emissions: direct emissions CH4                             (Year, Sector) float64 ...
    Emissions: direct emissions N2O                             (Year, Sector) float64 ...
    Emissions: direct emissions F-gases                         (Year, Sector) float64 ...
    Emissions: direct abatement total                           (Year, Sector) float64 ...
    ...                                                          ...
    Energy: gross demand ammonia                                (Year) float64 ...
    Energy: gross demand methanol                               (Year) float64 ...
    Energy: gross demand synmethanol                            (Year) float64 ...
    Energy: additional gross demand ammonia                     (Year) float64 ...
    Energy: additional gross demand methanol                    (Year) float64 ...
    Energy: additional gross demand synmethanol                 (Year) float64 ...

## Subsector data

In [6]:
subsector_level_data = pd.read_excel(datafile, sheet_name="Subsector-level data")
subsector_level_data.head()

,scenario,country,sector,subsector,variable,variable_unit,year,value
0,Balanced Pathway,United Kingdom,Agriculture,Enteric fermentation,Emissions: direct emissions total,MtCO2e,2025,23.452058
1,Balanced Pathway,United Kingdom,Agriculture,Enteric fermentation,Emissions: direct emissions total,MtCO2e,2026,23.306306
2,Balanced Pathway,United Kingdom,Agriculture,Enteric fermentation,Emissions: direct emissions total,MtCO2e,2027,22.673973
3,Balanced Pathway,United Kingdom,Agriculture,Enteric fermentation,Emissions: direct emissions total,MtCO2e,2028,21.897597
4,Balanced Pathway,United Kingdom,Agriculture,Enteric fermentation,Emissions: direct emissions total,MtCO2e,2029,21.009433


In [7]:
subsector_dataset = xr.Dataset()

for variable in subsector_level_data["variable"].unique():
    variable_data = subsector_level_data[subsector_level_data["variable"] == variable]
    variable_data = variable_data[variable_data["scenario"] == "Balanced Pathway"]
    variable_data = variable_data.drop(columns=["variable", "scenario", "country", "variable_unit", "sector"])
    variable_ds = xr.Dataset.from_dataframe(variable_data.set_index(["year", "subsector"]))
    subsector_dataset[variable] = variable_ds.to_array().squeeze()

subsector_dataset = subsector_dataset.drop("variable")
subsector_dataset = subsector_dataset.rename({"year": "Year", "subsector": "Subsector"})

In [8]:
subsector_dataset

<xarray.Dataset>
Dimensions:                                              (Year: 26,
                                                          Subsector: 59)
Coordinates:
  * Year                                                 (Year) int64 2025 .....
  * Subsector                                            (Subsector) object '...
Data variables: (12/36)
    Emissions: direct emissions total                    (Year, Subsector) float64 ...
    Emissions: direct emissions CO2                      (Year, Subsector) float64 ...
    Emissions: direct emissions CH4                      (Year, Subsector) float64 ...
    Emissions: direct emissions N2O                      (Year, Subsector) float64 ...
    Emissions: direct emissions F-gases                  (Year, Subsector) float64 ...
    Emissions: direct abatement total                    (Year, Subsector) float64 ...
    ...                                                   ...
    Energy: gross demand ammonia                         (Year, Subsector) float64 ...
    Energy: gross demand methanol                        (Year, Subsector) float64 ...
    Energy: gross demand synmethanol                     (Year, Subsector) float64 ...
    Energy: additional gross demand ammonia              (Year, Subsector) float64 ...
    Energy: additional gross demand methanol             (Year, Subsector) float64 ...
    Energy: additional gross demand synmethanol          (Year, Subsector) float64 ...

In [9]:
# We will also add a label coordinate to identify the sector each subsector belongs to
sector_names = subsector_level_data[["subsector", "sector"]].drop_duplicates().set_index("subsector")["sector"]
# subsector_dataset = subsector_dataset.assign_coords(Sector=("Subsector", subsector_dataset["Subsector"].values))
subsector_dataset = subsector_dataset.assign_coords(Sector=("Subsector", subsector_dataset["Subsector"].values))
subsector_dataset["Sector"] = subsector_dataset["Sector"].copy(data=sector_names.loc[subsector_dataset["Subsector"].values].values)
subsector_dataset


<xarray.Dataset>
Dimensions:                                              (Year: 26,
                                                          Subsector: 59)
Coordinates:
  * Year                                                 (Year) int64 2025 .....
  * Subsector                                            (Subsector) object '...
    Sector                                               (Subsector) object '...
Data variables: (12/36)
    Emissions: direct emissions total                    (Year, Subsector) float64 ...
    Emissions: direct emissions CO2                      (Year, Subsector) float64 ...
    Emissions: direct emissions CH4                      (Year, Subsector) float64 ...
    Emissions: direct emissions N2O                      (Year, Subsector) float64 ...
    Emissions: direct emissions F-gases                  (Year, Subsector) float64 ...
    Emissions: direct abatement total                    (Year, Subsector) float64 ...
    ...                                                   ...
    Energy: gross demand ammonia                         (Year, Subsector) float64 ...
    Energy: gross demand methanol                        (Year, Subsector) float64 ...
    Energy: gross demand synmethanol                     (Year, Subsector) float64 ...
    Energy: additional gross demand ammonia              (Year, Subsector) float64 ...
    Energy: additional gross demand methanol             (Year, Subsector) float64 ...
    Energy: additional gross demand synmethanol          (Year, Subsector) float64 ...

In [10]:
sector_dataset.to_netcdf("data/CB7_balanced_pathway_sector.nc")
subsector_dataset.to_netcdf("data/CB7_balanced_pathway_subsector.nc")